# 01 — Data Exploration: Physionet 2017 and Apple Watch
## Heartbeat or Noise? Evaluating Consumer Wearables as Cardiac Screening Proxies

---

### The Problem
Cardiovascular disease accounts for 30.5% of all deaths in Singapore — approximately 22 deaths every day. Yet 49% of Singapore residents did not have a health checkup in the past 12 months.

Consumer wearables are ubiquitous, passive, and already on people's wrists. This project asks one question: **can machine learning models trained on clinical ECG data generalise to consumer wearable signals to detect cardiac rhythm anomalies?**

If yes, wearables represent a scalable, low-barrier screening intervention for an under-screened population.

---

### What This Notebook Does
Explores and validates both datasets before any preprocessing begins. Two data sources:
- **Physionet 2017 AF Challenge** — 8,528 single-lead clinical ECG recordings. The training foundation.
- **Apple Watch (personal export)** — 4.8 years of personal wearable data including heart rate, HRV, sleep, and workouts. The exploratory bridge dataset.

No features are computed here. No model decisions are made. This notebook establishes what data exists and confirms it is usable.

## Physionet 2017 — Data Acquisition and Validation

In [ ]:
import os
import subprocess
import pandas as pd

PROJECT_ROOT = subprocess.check_output(
    ['git', 'rev-parse', '--show-toplevel'], text=True
).strip()

# Check ECG files
files = os.listdir(os.path.join(PROJECT_ROOT, 'data', 'physionet'))
mat_files = [f for f in files if f.endswith('.mat')]
print(f"Total ECG recordings: {len(mat_files)}")

# Check labels
labels = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'physionet', 'REFERENCE-v3.csv'), header=None, names=['record', 'label'])
print(f"\nTotal labels: {len(labels)}")
print("\nClass distribution:")
print(labels['label'].value_counts())

In [ ]:
import wfdb
import matplotlib.pyplot as plt

# Load a single record
record = wfdb.rdrecord(os.path.join(PROJECT_ROOT, 'data', 'physionet', 'A00001'))

print("Signal length:", record.sig_len)
print("Sampling frequency:", record.fs, "Hz")
print("Signal name:", record.sig_name)
print("Number of channels:", record.n_sig)
print("Duration (seconds):", record.sig_len / record.fs)

# Plot the raw ECG signal
plt.figure(figsize=(12, 4))
plt.plot(record.p_signal[:, 0])
plt.title('Raw ECG Signal - A00001')
plt.xlabel('Sample')
plt.ylabel('Amplitude (mV)')
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'figures', 'sample_ecg.png'), dpi=150, bbox_inches='tight')
plt.show()

---

## Apple Watch — Raw Export Validation

In [ ]:
import os

xml_path = os.path.join(PROJECT_ROOT, 'data', 'apple_watch', 'export.xml')

if os.path.exists(xml_path):
    size_gb = os.path.getsize(xml_path) / (1024**3)
    print(f"File found: {xml_path}")
    print(f"File size: {size_gb:.2f} GB")
else:
    print("FILE NOT FOUND — check the path and try again")
    files = os.listdir(os.path.join(PROJECT_ROOT, 'data', 'apple_watch'))
    print(f"\nFiles in apple_watch folder: {files}")

## Apple Watch — Data Type Discovery

Preview first 50,000 records from Apple Health XML to identify available data types without loading the full file.

In [ ]:
import xml.etree.ElementTree as ET

xml_path = os.path.join(PROJECT_ROOT, 'data', 'apple_watch', 'export.xml')

record_types = {}
count = 0

for event, elem in ET.iterparse(xml_path, events=('end',)):
    if elem.tag == 'Record':
        record_type = elem.attrib.get('type', 'unknown')
        if record_type not in record_types:
            record_types[record_type] = {
                'sample_record': dict(elem.attrib),
                'count': 0
            }
        record_types[record_type]['count'] += 1
        count += 1
        elem.clear()

    if count >= 50000:
        break

print(f"Data types found in first 50,000 records:\n")
for rtype, info in sorted(record_types.items()):
    print(f"Type: {rtype}")
    print(f"Count so far: {info['count']}")
    print(f"Sample attributes: {info['sample_record']}\n")

## Apple Watch — Source Device Timeline

Full parse of Apple Health XML to identify all unique source devices and their date ranges. Establishes which devices contributed data and when.

In [ ]:
import xml.etree.ElementTree as ET

xml_path = os.path.join(PROJECT_ROOT, 'data', 'apple_watch', 'export.xml')

sources = {}
total = 0

for event, elem in ET.iterparse(xml_path, events=('end',)):
    if elem.tag == 'Record':
        total += 1
        source = elem.attrib.get('sourceName', 'unknown')
        date = elem.attrib.get('startDate', '')[:10]

        if source not in sources:
            sources[source] = {
                'first_seen': date,
                'last_seen': date,
                'count': 0
            }
        else:
            if date < sources[source]['first_seen']:
                sources[source]['first_seen'] = date
            if date > sources[source]['last_seen']:
                sources[source]['last_seen'] = date
        sources[source]['count'] += 1

        elem.clear()

    if total % 500000 == 0:
        print(f"Processed {total:,} records...")

print(f"\nTotal records scanned: {total:,}")
print(f"\n--- ALL SOURCE DEVICES ---\n")
for source, info in sorted(sources.items(),
                           key=lambda x: x[1]['count'],
                           reverse=True):
    print(f"Source:     {source}")
    print(f"Date range: {info['first_seen']} to {info['last_seen']}")
    print(f"Records:    {info['count']:,}")
    print()

## Apple Watch — Full Extraction Pipeline

Single-pass extraction of all cardiac-relevant metrics from the Apple Health XML export. Filters exclusively to Apple Watch source data and saves separate `_raw.csv` files for each metric type.

**Device:** Apple Watch SE 1st Generation, Model A2352
**Coverage:** April 2021 — February 2026

**Target metrics:**
- Heart rate variability (SDNN)
- Heart rate (continuous)
- Resting heart rate
- Walking heart rate average
- Respiratory rate
- Sleep analysis
- VO2 Max
- Heart rate recovery (1 minute)
- Workouts

Raw files are unprocessed source extracts. Cleaning, timezone conversion, and outlier removal happen in `02_feature_engineering.ipynb` — not here.

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import os

"""
Apple Health XML — Full Extraction Pipeline
============================================
Device:      Apple Watch SE 1st Generation, Model A2352
Source file: export.xml (1.58 GB)
Coverage:    April 2021 — February 2026

Extracts all cardiac-relevant metrics in a single pass.
Filters exclusively to Apple Watch source data.
Preserves raw timestamps with Singapore +0800 offset.
Saves separate _raw.csv files for each metric type.

Raw files are unprocessed source extracts.
Cleaning, timezone conversion, and outlier removal
happen in 02_feature_engineering.ipynb — not here.
"""

# ── Configuration ────────────────────────────────────────────────

XML_PATH = os.path.join(PROJECT_ROOT, 'data', 'apple_watch', 'export.xml')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'apple_watch')

# Apple Health type identifiers — exact strings from XML
TARGET_TYPES = {
    'HKQuantityTypeIdentifierHeartRateVariabilitySDNN':
        'hrv',
    'HKQuantityTypeIdentifierHeartRate':
        'heart_rate',
    'HKQuantityTypeIdentifierRestingHeartRate':
        'resting_hr',
    'HKQuantityTypeIdentifierWalkingHeartRateAverage':
        'walking_hr',
    'HKQuantityTypeIdentifierRespiratoryRate':
        'respiratory_rate',
    'HKCategoryTypeIdentifierSleepAnalysis':
        'sleep',
    'HKQuantityTypeIdentifierVO2Max':
        'vo2max',
    'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute':
        'hr_recovery',
}

# Source filter — Apple Watch SE logs as "Lucas's Apple Watch"
# or "Apple Watch" depending on watchOS version
# We capture both to ensure no records are missed
WATCH_SOURCE_KEYWORDS = ['apple watch', 'watch']

# ── Initialise storage buckets ───────────────────────────────────

buckets = {name: [] for name in TARGET_TYPES.values()}
workout_records = []

total_records   = 0
watch_records   = 0
skipped_records = 0

# ── Single-pass XML parse ────────────────────────────────────────

print("="*60)
print("APPLE HEALTH XML EXTRACTION")
print("="*60)
print(f"Source: {XML_PATH}")
print(f"Output: {OUTPUT_DIR}")
print("\nParsing — do not interrupt. "
      "This will take 5-15 minutes...\n")

for event, elem in ET.iterparse(XML_PATH, events=('end',)):

    # ── Health records ───────────────────────────────────────────
    if elem.tag == 'Record':
        total_records += 1
        record_type = elem.attrib.get('type', '')

        if record_type in TARGET_TYPES:
            source = elem.attrib.get('sourceName', '').lower()

            # Filter to Watch source only
            is_watch = any(
                kw in source for kw in WATCH_SOURCE_KEYWORDS
            )

            if not is_watch:
                skipped_records += 1
                elem.clear()
                continue

            watch_records += 1
            bucket_name = TARGET_TYPES[record_type]

            buckets[bucket_name].append({
                'startDate':     elem.attrib.get('startDate'),
                'endDate':       elem.attrib.get('endDate'),
                'value':         elem.attrib.get('value'),
                'unit':          elem.attrib.get('unit'),
                'sourceName':    elem.attrib.get('sourceName'),
                'sourceVersion': elem.attrib.get(
                                     'sourceVersion', ''),
            })

    # ── Workout records ──────────────────────────────────────────
    elif elem.tag == 'Workout':
        source = elem.attrib.get('sourceName', '').lower()
        is_watch = any(
            kw in source for kw in WATCH_SOURCE_KEYWORDS
        )
        if is_watch:
            workout_records.append({
                'workoutType': elem.attrib.get(
                                   'workoutActivityType'),
                'startDate':   elem.attrib.get('startDate'),
                'endDate':     elem.attrib.get('endDate'),
                'duration':    elem.attrib.get('duration'),
                'durationUnit':elem.attrib.get('durationUnit'),
                'sourceName':  elem.attrib.get('sourceName'),
            })

    # Free memory after each element
    elem.clear()

    # Progress indicator
    if total_records % 500000 == 0:
        print(f"  Processed {total_records:,} records...")

print(f"\nParse complete.")
print(f"  Total records scanned:  {total_records:,}")
print(f"  Watch records kept:     {watch_records:,}")
print(f"  Non-watch records skipped: {skipped_records:,}")

# ── Save raw CSV files ───────────────────────────────────────────

print("\n" + "="*60)
print("SAVING RAW EXTRACTION FILES")
print("="*60)

extraction_summary = {}

for metric_name, records in buckets.items():
    filename = f'{metric_name}_raw.csv'
    filepath = os.path.join(OUTPUT_DIR, filename)

    if records:
        df = pd.DataFrame(records)
        df.to_csv(filepath, index=False)
        extraction_summary[metric_name] = {
            'count':    len(df),
            'status':   'EXTRACTED',
            'filename': filename
        }
        print(f"  SAVED    {filename} — {len(df):,} records")
    else:
        extraction_summary[metric_name] = {
            'count':    0,
            'status':   'NOT FOUND',
            'filename': filename
        }
        print(f"  MISSING  {filename} — 0 records found")

# Workouts
if workout_records:
    df_workouts = pd.DataFrame(workout_records)
    workout_path = os.path.join(OUTPUT_DIR, 'workouts_raw.csv')
    df_workouts.to_csv(workout_path, index=False)
    extraction_summary['workouts'] = {
        'count':    len(df_workouts),
        'status':   'EXTRACTED',
        'filename': 'workouts_raw.csv'
    }
    print(f"  SAVED    workouts_raw.csv — "
          f"{len(df_workouts):,} records")
else:
    extraction_summary['workouts'] = {
        'count':    0,
        'status':   'NOT FOUND',
        'filename': 'workouts_raw.csv'
    }
    print(f"  MISSING  workouts_raw.csv — 0 records found")

# ── Final summary ────────────────────────────────────────────────

print("\n" + "="*60)
print("EXTRACTION SUMMARY")
print("="*60)
print(f"{'METRIC':<35} {'STATUS':<12} {'RECORDS':>10}")
print("-"*60)

for metric, info in extraction_summary.items():
    print(f"{metric:<35} "
          f"{info['status']:<12} "
          f"{info['count']:>10,}")

print("="*60)
print("\nExtraction complete.")
print("Next step: 02_feature_engineering.ipynb")
print("Do not modify raw CSV files — "
      "they are your source of truth.")

In [ ]:
import os
import pandas as pd

OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'apple_watch')

expected_files = [
    'hrv_raw.csv',
    'heart_rate_raw.csv',
    'resting_hr_raw.csv',
    'walking_hr_raw.csv',
    'respiratory_rate_raw.csv',
    'sleep_raw.csv',
    'vo2max_raw.csv',
    'hr_recovery_raw.csv',
    'workouts_raw.csv'
]

print(f"{'FILE':<35} {'STATUS':<12} {'RECORDS':>10} {'SIZE':>10}")
print("-"*70)

for filename in expected_files:
    filepath = os.path.join(OUTPUT_DIR, filename)
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"{filename:<35} {'FOUND':<12} "
              f"{len(df):>10,} {size_kb:>8.1f} KB")
    else:
        print(f"{filename:<35} {'MISSING':<12} "
              f"{'—':>10} {'—':>10}")

print("-"*70)
print("\nTimestamp sample from hrv_raw.csv:")
hrv_path = os.path.join(OUTPUT_DIR, 'hrv_raw.csv')
if os.path.exists(hrv_path):
    hrv = pd.read_csv(hrv_path)
    print(hrv[['startDate', 'value', 
               'sourceName']].head(3).to_string())

### Data Confirmed
Both datasets are validated and ready for preprocessing.

- **Physionet:** 8,528 ECG recordings (8,249 after noisy class exclusion), sampled at 300 Hz, with clinical labels: Normal (5,076), AF (758), Other (2,415).
- **Apple Watch:** 9 raw CSV files extracted from personal export — 4.8 years of heart rate, HRV, resting HR, walking HR, respiratory rate, sleep, and workout records.

**Next:** Notebook 02 cleans both datasets and extracts the 8 features that will be used in all subsequent modelling.

In [ ]:
metrics_to_clean = [
    'hrv', 'heart_rate', 'resting_hr',
    'walking_hr', 'respiratory_rate'
]

